In [2]:
from fyers_apiv3 import fyersModel


CLIENT_ID = 'G80JG7X1EA-200'

with open('access_token.txt') as f:
    access_token = f.read().strip()

fyers = fyersModel.FyersModel(client_id=CLIENT_ID, token=access_token, is_async=False, log_path='')



In [ ]:

def place_market_order(symbol, qty, side='BUY', product_type='INTRADAY'):
    data = {
        'symbol': symbol,
        'qty': qty,
        'type': 2,                              # 1=Limit, 2=Market, 3=Stop(SL-M), 4=Stoplimit(SL-L)
        'side': 1 if side == 'BUY' else -1,     # 1=Buy, -1=Sell
        'productType': product_type,            # INTRADAY, CNC, MARGIN, CO, BO
        'limitPrice': 0,
        'stopPrice': 0,
        'validity': 'DAY',
        'disclosedQty': 0,
        'offlineOrder': False,                  # True for AMO (after market hours)
        'stopLoss': 0,
        'takeProfit': 0,
        'orderTag': 'idea1',
    }
    response = fyers.place_order(data=data)
    if response.get('s') != 'ok':
        raise SystemExit(f'Order failed: {response}')

    print(f"Order placed: {side} {qty} {symbol}  ->  Order ID: {response['id']}")
    return response


place_market_order('NSE:IDEA-EQ', qty=1, side='BUY')


Order placed: BUY 1 NSE:IDEA-EQ  ->  Order ID: 26100500381526


{'code': 1101,
 'message': 'Successfully placed order',
 's': 'ok',
 'id_fyers': '336e871c-8e98-400a-b878-b69c464b3eab',
 'id': '26100500381526'}

In [3]:
def place_limit_order(symbol, qty, limit_price, side='BUY', product_type='INTRADAY'):
    data = {
        'symbol': symbol,
        'qty': qty,
        'type': 1,                              # 1=Limit, 2=Market, 3=Stop(SL-M), 4=Stoplimit(SL-L)
        'side': 1 if side == 'BUY' else -1,     # 1=Buy, -1=Sell
        'productType': product_type,            # INTRADAY, CNC, MARGIN, CO, BO
        'limitPrice': limit_price,
        'stopPrice': 0,
        'validity': 'DAY',
        'disclosedQty': 0,
        'offlineOrder': False,                  # True for AMO (after market hours)
        'stopLoss': 0,
        'takeProfit': 0,
        'orderTag': 'idealimit1',
    }
    response = fyers.place_order(data=data)
    if response.get('s') != 'ok':
        raise SystemExit(f'Order failed: {response}')

    print(f"Limit order placed: {side} {qty} {symbol} @ {limit_price}  ->  Order ID: {response['id']}")
    return response


# Buy 1 share of IDEA at 1% below the current LTP (rounded to the 0.01 tick size)
quote = fyers.quotes(data={'symbols': 'NSE:IDEA-EQ'})
ltp = quote['d'][0]['v']['lp']
limit_price = round(13, 2)
print(f'LTP: {ltp}   Limit price: {limit_price}')

place_limit_order('NSE:IDEA-EQ', qty=1, limit_price=limit_price, side='BUY')


LTP: 13.18   Limit price: 13
Limit order placed: BUY 1 NSE:IDEA-EQ @ 13  ->  Order ID: 26100700337284


{'code': 1101,
 'message': 'Successfully placed order',
 's': 'ok',
 'id_fyers': 'ffd48590-e3a4-4f6b-9712-bb6d9ac5603a',
 'id': '26100700337284'}

In [9]:
def place_stop_loss_order(symbol, qty, stop_price, limit_price=0, side='SELL', product_type='INTRADAY'):
    # limit_price=0 -> SL-M (type 3): becomes a market order once stop_price is hit
    # limit_price>0 -> SL-L (type 4): becomes a limit order at limit_price once stop_price is hit
    data = {
        'symbol': symbol,
        'qty': qty,
        'type': 4 if limit_price else 3,        # 1=Limit, 2=Market, 3=Stop(SL-M), 4=Stoplimit(SL-L)
        'side': 1 if side == 'BUY' else -1,     # 1=Buy, -1=Sell
        'productType': product_type,            # INTRADAY, CNC, MARGIN, CO, BO
        'limitPrice': limit_price,
        'stopPrice': stop_price,                # trigger price
        'validity': 'DAY',
        'disclosedQty': 0,
        'offlineOrder': False,                  # True for AMO (after market hours)
        'stopLoss': 0,
        'takeProfit': 0,
        'orderTag': 'ideasl1',
    }
    response = fyers.place_order(data=data)
    if response.get('s') != 'ok':
        raise SystemExit(f'Order failed: {response}')

    kind = 'SL-L' if limit_price else 'SL-M'
    print(f"{kind} order placed: {side} {qty} {symbol} trigger {stop_price} limit {limit_price}  ->  Order ID: {response['id']}")
    return response


# Stop loss to protect a bought IDEA position: sell if price falls 2% below LTP
quote = fyers.quotes(data={'symbols': 'NSE:IDEA-EQ'})
ltp = quote['d'][0]['v']['lp']
stop_price = round(12, 2)
limit_price = round(11.99, 2)   # for a SELL SL-L, limit must be at or below the trigger
print(f'LTP: {ltp}   Trigger: {stop_price}   Limit: {limit_price}')

# SL-L (stop limit)
place_stop_loss_order('NSE:IDEA-EQ', qty=1, stop_price=stop_price, limit_price=limit_price, side='SELL')

# SL-M (stop market) instead:
# place_stop_loss_order('NSE:IDEA-EQ', qty=1, stop_price=stop_price, side='SELL')


LTP: 12.77   Trigger: 12   Limit: 11.99
SL-L order placed: SELL 1 NSE:IDEA-EQ trigger 12 limit 11.99  ->  Order ID: 26100500401413


{'code': 1101,
 'message': 'Successfully placed order',
 's': 'ok',
 'id_fyers': 'adc672f8-392e-45ad-93a5-d2b3ac090121',
 'id': '26100500401413'}

In [4]:
import pandas as pd


def get_orders_df(save_path='orders.csv'):
    response = fyers.orderbook()
    if response.get('s') != 'ok':
        raise SystemExit(f'Failed to fetch orders: {response}')

    orders = response.get('orderBook', [])
    if not orders:
        print('No orders found for today')
        return pd.DataFrame()

    df = pd.DataFrame(orders)

    # Human readable labels for the numeric codes
    status_map = {1: 'CANCELLED', 2: 'FILLED', 4: 'TRANSIT', 5: 'REJECTED', 6: 'PENDING', 7: 'EXPIRED'}
    type_map = {1: 'LIMIT', 2: 'MARKET', 3: 'SL-M', 4: 'SL-L'}
    df['statusText'] = df['status'].map(status_map)
    df['typeText'] = df['type'].map(type_map)
    df['sideText'] = df['side'].map({1: 'BUY', -1: 'SELL'})

    df.to_csv(save_path, index=False)
    print(f'Saved {len(df)} orders to {save_path}')
    return df


orders_df = get_orders_df()
cols = ['id', 'symbol', 'sideText', 'typeText', 'qty', 'filledQty', 'limitPrice', 'stopPrice',
        'tradedPrice', 'statusText', 'productType', 'orderDateTime', 'message']
orders_df[[c for c in cols if c in orders_df.columns]]


Saved 1 orders to orders.csv


,id,symbol,sideText,typeText,qty,filledQty,limitPrice,stopPrice,tradedPrice,statusText,productType,orderDateTime,message
0,26100700337284,NSE:IDEA-EQ,BUY,LIMIT,1,0,13,0,0,PENDING,INTRADAY,07-Oct-2026 13:46:14,


In [5]:
def modify_limit_price(symbol, new_price):
    response = fyers.orderbook()
    if response.get('s') != 'ok':
        raise SystemExit(f'Failed to fetch orders: {response}')

    # Pending (status 6) limit orders (type 1) for this symbol
    pending = [o for o in response.get('orderBook', [])
               if o['symbol'] == symbol and o['type'] == 1 and o['status'] == 6]
    if not pending:
        print(f'No pending limit order found for {symbol}')
        return None

    for order in pending:
        data = {
            'id': order['id'],
            'type': 1,                  # keep it a limit order
            'limitPrice': new_price,
            'qty': order['qty'],
        }
        result = fyers.modify_order(data=data)
        if result.get('s') != 'ok':
            print(f"Modify failed for {order['id']}: {result}")
        else:
            print(f"Order {order['id']} modified: {order['limitPrice']} -> {new_price}")
    return pending


modify_limit_price('NSE:IDEA-EQ', new_price=13.1)


Order 26100700337284 modified: 13 -> 13.1


[{'clientId': 'XS45474',
  'exchange': 10,
  'fyToken': '101000000014366',
  'id': '26100700337284',
  'id_fyers': 'ffd48590-e3a4-4f6b-9712-bb6d9ac5603a',
  'instrument': 0,
  'offlineOrder': False,
  'source': 'API',
  'status': 6,
  'type': 1,
  'pan': 'GXAPS1597B',
  'limitPrice': 13,
  'productType': 'INTRADAY',
  'qty': 1,
  'disclosedQty': 0,
  'remainingQuantity': 1,
  'segment': 10,
  'symbol': 'NSE:IDEA-EQ',
  'description': 'VODAFONE IDEA LIMITED',
  'ex_sym': 'IDEA',
  'orderDateTime': '07-Oct-2026 13:46:14',
  'side': 1,
  'orderValidity': 'DAY',
  'stopPrice': 0,
  'tradedPrice': 0,
  'filledQty': 0,
  'exchOrdId': '1100000068238952',
  'message': '',
  'ch': -0.03,
  'chp': -0.22710068130204392,
  'lp': 13.18,
  'orderNumStatus': '26100700337284:6',
  'slNo': 1,
  'orderTag': '1:idealimit1',
  'takeProfit': 0,
  'stopLoss': 0}]

In [6]:
def cancel_pending_orders(symbol):
    response = fyers.orderbook()
    if response.get('s') != 'ok':
        raise SystemExit(f'Failed to fetch orders: {response}')

    # Only pending orders (status 6) can be cancelled
    pending = [o for o in response.get('orderBook', [])
               if o['symbol'] == symbol and o['status'] == 6]
    if not pending:
        print(f'No pending order found for {symbol}')
        return None

    for order in pending:
        result = fyers.cancel_order(data={'id': order['id']})
        if result.get('s') != 'ok':
            print(f"Cancel failed for {order['id']}: {result}")
        else:
            print(f"Order {order['id']} cancelled ({order['qty']} {symbol} @ {order['limitPrice']})")
    return pending


cancel_pending_orders('NSE:IDEA-EQ')


Order 26100700337284 cancelled (1 NSE:IDEA-EQ @ 13.1)


[{'clientId': 'XS45474',
  'exchange': 10,
  'fyToken': '101000000014366',
  'id': '26100700337284',
  'id_fyers': 'ffd48590-e3a4-4f6b-9712-bb6d9ac5603a',
  'instrument': 0,
  'offlineOrder': False,
  'source': 'API',
  'status': 6,
  'type': 1,
  'pan': 'GXAPS1597B',
  'limitPrice': 13.1,
  'productType': 'INTRADAY',
  'qty': 1,
  'disclosedQty': 0,
  'remainingQuantity': 1,
  'segment': 10,
  'symbol': 'NSE:IDEA-EQ',
  'description': 'VODAFONE IDEA LIMITED',
  'ex_sym': 'IDEA',
  'orderDateTime': '07-Oct-2026 13:51:06',
  'side': 1,
  'orderValidity': 'DAY',
  'stopPrice': 0,
  'tradedPrice': 0,
  'filledQty': 0,
  'exchOrdId': '1100000068238952',
  'message': '',
  'ch': -0.03,
  'chp': -0.22710068130204392,
  'lp': 13.18,
  'orderNumStatus': '26100700337284:6',
  'slNo': 1,
  'orderTag': '1:idealimit1',
  'takeProfit': 0,
  'stopLoss': 0}]

In [7]:
def place_gtt_limit_order(symbol, qty, trigger_price, limit_price, side='BUY', product_type='CNC'):
    # GTT stays active across days until the trigger is hit (or it is cancelled)
    # product_type for GTT must be CNC, MARGIN or MTF (INTRADAY is not allowed)
    data = {
        'side': 1 if side == 'BUY' else -1,     # 1=Buy, -1=Sell
        'symbol': symbol,
        'productType': product_type,
        'orderInfo': {
            'leg1': {
                'price': limit_price,           # limit price of the order placed on trigger
                'triggerPrice': trigger_price,  # order is sent to exchange when LTP hits this
                'qty': qty,
            }
        },
    }
    response = fyers.place_gtt_order(data=data)
    if response.get('s') != 'ok':
        raise SystemExit(f'GTT order failed: {response}')

    print(f"GTT order placed: {side} {qty} {symbol} trigger {trigger_price} limit {limit_price}  ->  ID: {response.get('id')}")
    return response


# Buy 1 IDEA when price comes down to 12, with a limit price of 12
place_gtt_limit_order('NSE:IDEA-EQ', qty=1, trigger_price=12, limit_price=12, side='BUY')

# View all GTT orders
# fyers.gtt_orderbook()


GTT order placed: BUY 1 NSE:IDEA-EQ trigger 12 limit 12  ->  ID: 26100700001089


{'code': 1101,
 'message': 'Successfully placed order',
 's': 'ok',
 'id_fyers': '7dc7befc-5bc5-44ae-bbf4-acfc62d24cb6',
 'id': '26100700001089'}